#XOR Gate Backpropagation (Batch Gradient Descent)
XOR 다층 퍼셉트론(MLP) 코드는 입력층(2개 노드) $\rightarrow$ 은닉층(2개 노드) $\rightarrow$ 출력층(1개 노드)으로 구성된 신경망을 미니배치 크기 $N=4$ 로 한 번에 학습하는 구조입니다.<br>역전파(Backpropagation)의 핵심은 손실 함수 $L$ 에 대한 각 가중치와 편향의 기울기(Gradient)를 연쇄 법칙(Chain Rule)으로 구하는 것이며, 미니배치 행렬 연산에서 원래 변수의 행렬 모양(Shape)과 기울기의 모양을 일치시키고 각 데이터 샘플의 기울기를 합산하기 위해 전치 행렬(Transpose)이 필수적으로 사용됩니다.

###1. 변수별 행렬 차원(Shape) 및 기본 공식 정리
역전파와 전치 행렬을 이해하기 위해 각 변수의 차원(Shape)과 미분 전제 공식을 먼저 확인합니다. 미분의 기본 원칙은 어떤 행렬 $M$ 에 대한 스칼라 손실 $L$ 의 기울기 행렬 $\frac{\partial L}{\partial M}$ 은 반드시 원래 행렬 $M$ 과 차원이 완전히 같아야 한다는 점입니다(그래야 W -= lr * dL_dW 연산이 가능합니다).
|구분|변수명|코드 표현|행렬 차원 (Shape)|설명|
| --- | --- | --- | --- | --- |
|입력/정답|$X$, $y$|X, y|$(4, 2)$, $(4, 1)$|데이터 샘플 수 N=4|
|은닉층 파라미터|$W_1$, $b_1$|W1, b1|$(2, 2)$, $(1, 2)$|입력 2차원 → 은닉 2차원|
|은닉층 출력|$z_1$, $a_1$|z1, a1|$(4, 2)$, $(4, 2)$|선형 결합 결과 및 Sigmoid 통과 값|
|출력층 파라미터|$W_2$, $b_2$|W2, b2|$(2, 1)$, $(1, 1)$|은닉 2차원 → 출력 1차원|
|출력층 출력|$z_2$, $\hat{y}$|z2, y_hat|$(4, 1)$, $(4, 1)$|선형 결합 결과 및 최종 예측 확률|
* Sigmoid 함수의 미분: $\sigma(x) = \frac{1}{1 + e^{-x}}$ 일 때, 도함수는 $\frac{d\sigma(x)}{dx} = \sigma(x)(1 - \sigma(x))$ 입니다.
* 손실 함수 $L$ (Binary Cross Entropy):$$L = -\frac{1}{N} \sum_{i=1}^{N} \left[ y^{(i)} \ln(\hat{y}^{(i)}) + (1 - y^{(i)}) \ln(1 - \hat{y}^{(i)}) \right]$$

In [1]:
import numpy as np

np.random.seed(42)

# 1. XOR 데이터 준비
# X Shape: (4, 2) -> 샘플 수 N=4, 입력 특성 수 D_in=2
X = np.array([[0, 0], [0, 1], [1, 0], [1, 1]], dtype=float)

# y Shape: (4, 1) -> 샘플 수 N=4, 출력 정답 수 D_out=1
y = np.array([[0], [1], [1], [0]], dtype=float)

N = len(X)  # 미니배치 샘플 수 N = 4

# 2. Sigmoid 함수 정의
# 수식: sigma(x) = 1 / (1 + e^(-x))
# 도함수(미분): d_sigma(x) / dx = sigma(x) * (1 - sigma(x))
def sigmoid(x):
    return 1 / (1 + np.exp(-x))


# 3. 가중치 및 편향 초기화 (입력층 2 -> 은닉층 2 -> 출력층 1)
# 역전파의 기본 원칙: 어떤 행렬 M에 대한 기울기 dL/dM의 차원(Shape)은 반드시 원래 행렬 M과 같아야 함
W1 = np.array([ [ 4.0, 4.0], [4.0, 4.0] ]) # Shape: (2, 2) -> (입력 차원 2, 은닉 차원 2)
# W1 = np.random.randn(2, 2)
b1 = np.array([ [-2.0, -6.0] ]) # Shape: (1, 2) -> (1, 은닉 차원 2)
# b1 = np.zeros((1, 2))

W2 = np.array([ [ 4.0], [-8.0] ]) # Shape: (2, 1) -> (은닉 차원 2, 출력 차원 1)
# W2 = np.random.randn(2, 1)
b2 = np.array([ [-2.0] ]) # Shape: (1, 1) -> (1, 출력 차원 1)
# b2 = np.zeros((1, 1))

lr = 1.0  # 학습률 (Learning Rate)

### 2. 왜 전치 행렬(Transpose, .T)을 사용해야 하는가?
선형 변환 $Z = A W + b$ (예: $z_2 = a_1 W_2 + b_2$)에서 상위 층으로부터 오차 기울기 $\frac{\partial L}{\partial Z}$ 가 전달되었을 때, $\frac{\partial L}{\partial W}$ 와 $\frac{\partial L}{\partial A}$ 를 구할 때 전치 행렬이 등장하는 이유는 스칼라 연쇄 법칙의 합산($\sum$)을 행렬 곱으로 한 번에 처리하고 차원(Shape)을 맞추기 위함입니다.<br><br>
일반화된 차원 $A \in \mathbb{R}^{N \times D_{in}}$, $W \in \mathbb{R}^{D_{in} \times D_{out}}$, $Z \in \mathbb{R}^{N \times D_{out}}$ 를 가정해 보겠습니다. $i$ 번째 샘플의 $k$ 번째 출력 원소는 다음과 같습니다.$$Z_{i, k} = \sum_{j=1}^{D_{in}} A_{i, j} W_{j, k} + b_{1, k}$$
<br>
  ① 가중치 미분 ($\frac{\partial L}{\partial W} = A^T \frac{\partial L}{\partial Z}$)에서 $A^T$ 를 쓰는 이유가중치 행렬의 특정 원소 $W_{j, k}$ 는 미니배치 내 모든 $N$ 개 데이터 샘플의 $k$ 번째 출력 $Z_{1,k}, Z_{2,k}, \dots, Z_{N,k}$ 계산에 전부 관여합니다. 따라서 연쇄 법칙에 의해 $W_{j, k}$ 에 대한 미분은 모든 샘플 $i=1 \dots N$ 에 대한 영향을 모두 더해야 합니다.
$$\frac{\partial L}{\partial W_{j, k}} = \sum_{i=1}^{N} \frac{\partial L}{\partial Z_{i, k}} \cdot \frac{\partial Z_{i, k}}{\partial W_{j, k}} = \sum_{i=1}^{N} \frac{\partial L}{\partial Z_{i, k}} \cdot A_{i, j} = \sum_{i=1}^{N} (A^T)_{j, i} \left(\frac{\partial L}{\partial Z}\right)_{i, k}$$
<br>
* 원소 관점: 샘플 인덱스 $i$ 를 기준으로 내적(합산)하려면 $A_{i, j}$ 의 행과 열을 뒤집어 $(A^T)_{j, i}$ 로 만들어야 행렬 곱의 정의(앞 행렬의 열 인덱스 $i$ = 뒤 행렬의 행 인덱스 $i$)가 성립합니다.
* 차원(Shape) 관점: 구해야 하는 $\frac{\partial L}{\partial W}$ 의 차원은 $(D_{in}, D_{out})$ 입니다. 가진 재료는 $A$ $(N, D_{in})$ 와 $\frac{\partial L}{\partial Z}$ $(N, D_{out})$ 이므로, 미니배치 차원 $N$ 을 소거하고 $(D_{in}, D_{out})$ 을 만들려면 $A$ 를 전치시켜 $(D_{in}, N) \times (N, D_{out}) = (D_{in}, D_{out})$ 형태로 곱해야만 합니다.
<br>
  ② 입력(이전 층 활성화 값) 미분 ($\frac{\partial L}{\partial A} = \frac{\partial L}{\partial Z} W^T$)에서 $W^T$ 를 쓰는 이유특정 샘플 $i$ 의 은닉층 출력 원소 $A_{i, j}$ 는 다음 층의 모든 $D_{out}$ 개 출력 노드 $Z_{i, 1}, Z_{i, 2}, \dots, Z_{i, D_{out}}$ 로 퍼져 나갑니다. 따라서 역전파 시에는 다음 층의 모든 출력 노드로부터 돌아오는 오차를 다시 합산해야 합니다.

$$\frac{\partial L}{\partial A_{i, j}} = \sum_{k=1}^{D_{out}} \frac{\partial L}{\partial Z_{i, k}} \cdot \frac{\partial Z_{i, k}}{\partial A_{i, j}} = \sum_{k=1}^{D_{out}} \frac{\partial L}{\partial Z_{i, k}} \cdot W_{j, k} = \sum_{k=1}^{D_{out}} \left(\frac{\partial L}{\partial Z}\right)_{i, k} (W^T)_{k, j}$$
<br>
* 원소 관점: 순전파에서는 입력에서 출력 방향($j \rightarrow k$)으로 가중치 $W_{j, k}$ 를 곱했으므로, 역전파에서는 출력에서 입력 방향($k \rightarrow j$)으로 되돌아오기 위해 가중치 행렬의 방향을 뒤집은 $(W^T)_{k, j}$ 를 곱해 $k$ 에 대해 합산합니다.
* 차원(Shape) 관점: 구해야 하는 $\frac{\partial L}{\partial A}$ 의 차원은 $(N, D_{in})$ 입니다. $\frac{\partial L}{\partial Z}$ $(N, D_{out})$ 와 $W$ $(D_{in}, D_{out})$ 로부터 출력 차원 $D_{out}$ 을 소거하려면 $W$ 를 전치시켜 $(N, D_{out}) \times (D_{out}, D_{in}) = (N, D_{in})$ 형태로 곱해야 합니다.

### 3. 단계별 Backpropagation 미분 과정 상세 유도
#### Step 1: 출력층 선형 출력 $z_2$ 에 대한 손실 미분 (dL_dz2)
* 대상 코드:
  ```python
  dL_dz2 = (y_hat - y) / N
  ```
* 연쇄 법칙(Chain Rule) 명시:손실 $L$ 은 $\hat{y}$ 의 함수이고, $\hat{y} = \sigma(z_2)$ 는 $z_2$ 의 함수입니다. 각 샘플 $i$ 에 대해 스칼라 연쇄 법칙을 적용합니다(원소별 연산이므로 행렬 전체에도 동일하게 적용됩니다).$$\frac{\partial L}{\partial z_2^{(i)}} = \frac{\partial L}{\partial \hat{y}^{(i)}} \cdot \frac{\partial \hat{y}^{(i)}}{\partial z_2^{(i)}}$$
* 단계별 유도:
  1. $\frac{\partial L}{\partial \hat{y}^{(i)}}$ 계산: 손실 함수 $L = -\frac{1}{N} \sum \left[ y^{(i)} \ln(\hat{y}^{(i)}) + (1 - y^{(i)}) \ln(1 - \hat{y}^{(i)}) \right]$ 를 $\hat{y}^{(i)}$ 로 편미분합니다.$$\frac{\partial L}{\partial \hat{y}^{(i)}} = -\frac{1}{N} \left( \frac{y^{(i)}}{\hat{y}^{(i)}} - \frac{1 - y^{(i)}}{1 - \hat{y}^{(i)}} \right) = -\frac{1}{N} \left( \frac{y^{(i)}(1 - \hat{y}^{(i)}) - (1 - y^{(i)})\hat{y}^{(i)}}{\hat{y}^{(i)}(1 - \hat{y}^{(i)})} \right) = \frac{1}{N} \left( \frac{\hat{y}^{(i)} - y^{(i)}}{\hat{y}^{(i)}(1 - \hat{y}^{(i)})} \right)$$
  2. $\frac{\partial \hat{y}^{(i)}}{\partial z_2^{(i)}}$ 계산: $\hat{y}^{(i)} = \sigma(z_2^{(i)})$ 이므로 Sigmoid 도함수 공식을 적용합니다.$$\frac{\partial \hat{y}^{(i)}}{\partial z_2^{(i)}} = \hat{y}^{(i)}(1 - \hat{y}^{(i)})$$
  3. 두 식의 결합: 분모와 분자의 $\hat{y}^{(i)}(1 - \hat{y}^{(i)})$ 가 약분되어 깔끔하게 정리됩니다.$$\frac{\partial L}{\partial z_2^{(i)}} = \frac{1}{N} \left( \frac{\hat{y}^{(i)} - y^{(i)}}{{\hat{y}^{(i)}(1 - \hat{y}^{(i)})}} \right) \cdot {\hat{y}^{(i)}(1 - \hat{y}^{(i)})} = \frac{\hat{y}^{(i)} - y^{(i)}}{N}$$
* 행렬 표현 및 차원: 전체 샘플에 대해 행렬로 묶으면 $\frac{\partial L}{\partial z_2} = \frac{\hat{y} - y}{N}$ 이 되며, 차원은 $(4, 1)$ 입니다.
---
#### Step 2: 출력층 가중치 $W_2$ 와 편향 $b_2$ 미분 (dL_dW2, dL_db2)
* 대상 코드:
  ```python
  dL_dW2 = a1.T @ dL_dz2
  dL_db2 = np.sum(dL_dz2, axis=0, keepdims=True)
  ```
* 연쇄 법칙(Chain Rule) 명시:
순전파 식 $z_2 = a_1 W_2 + b_2$ (원소 형태: $z_{2}^{(i)} = \sum_{j=1}^{2} a_{1, j}^{(i)} W_{2, j, 1} + b_{2, 1}$)에 대하여 연쇄 법칙을 적용합니다.
$$\frac{\partial L}{\partial W_{2, j, 1}} = \sum_{i=1}^{N} \frac{\partial L}{\partial z_2^{(i)}} \cdot \frac{\partial z_2^{(i)}}{\partial W_{2, j, 1}}, \qquad \frac{\partial L}{\partial b_{2, 1}} = \sum_{i=1}^{N} \frac{\partial L}{\partial z_2^{(i)}} \cdot \frac{\partial z_2^{(i)}}{\partial b_{2, 1}}$$

* 단계별 유도 및 전치 행렬 사용:
  1. 가중치 $W_2$ 미분 (dL_dW2): <br>
  $\frac{\partial z_2^{(i)}}{\partial W_{2, j, 1}} = a_{1, j}^{(i)}$ 이므로, $\frac{\partial L}{\partial W_{2, j, 1}} = \sum_{i=1}^{N} a_{1, j}^{(i)} \frac{\partial L}{\partial z_2^{(i)}}$ 입니다.
  앞서 설명한 대로 4개의 샘플($i=1\dots 4$)에 대한 값을 곱해서 더하고, $W_2$ 의 원래 차원인 $(2, 1)$ 을 맞추기 위해 $(4, 2)$ 행렬인 a1을 전치시켜 $(2, 4)$ 인 a1.T로 만든 뒤 $(4, 1)$ 인 dL_dz2와 행렬 곱(@)을 수행합니다.
  $$\frac{\partial L}{\partial W_2} = a_1^T \frac{\partial L}{\partial z_2} \quad \Longrightarrow \quad \text{Shape: } (2, 4) \times (4, 1) = (2, 1)$$
  2. 편향 $b_2$ 미분 (dL_db2): <br>
  순전파 시 $b_2$ $(1, 1)$ 는 브로드캐스팅(Broadcasting)되어 4개의 샘플에 똑같이 더해졌습니다. 따라서 $\frac{\partial z_2^{(i)}}{\partial b_{2, 1}} = 1$ 이며, 역전파 시에는 4개 샘플에서 발생한 오차 $\frac{\partial L}{\partial z_2^{(i)}}$ 를 행 방향(axis=0, 샘플 방향)으로 모두 더해야 합니다(np.sum). keepdims=True를 통해 $b_2$ 의 원래 차원인 $(1, 1)$ 을 유지합니다.
---
  #### Step 3: 은닉층 활성화 출력 $a_1$ 및 선형 출력 $z_1$ 미분 (dL_da1, dL_dz1)
* 대상 코드:
  ```python
  dL_da1 = dL_dz2 @ W2.T
  dL_dz1 = dL_da1 * a1 * (1-a1)
  ```
* 연쇄 법칙(Chain Rule) 명시:
은닉층으로 오차를 역전파하는 과정입니다. 먼저 $z_2$ 를 통해 $a_1$ 의 미분을 구하고, 이어서 $a_1 = \sigma(z_1)$ 관계를 통해 $z_1$ 의 미분을 구합니다.
$$\frac{\partial L}{\partial a_{1, j}^{(i)}} = \sum_{k=1}^{1} \frac{\partial L}{\partial z_{2, k}^{(i)}} \cdot \frac{\partial z_{2, k}^{(i)}}{\partial a_{1, j}^{(i)}}, \qquad \frac{\partial L}{\partial z_{1, j}^{(i)}} = \frac{\partial L}{\partial a_{1, j}^{(i)}} \cdot \frac{\partial a_{1, j}^{(i)}}{\partial z_{1, j}^{(i)}}$$
* 단계별 유도 및 전치 행렬 사용:
  1. 은닉층 활성화 값 $a_1$ 미분 (dL_da1): <br>
  $z_2 = a_1 W_2 + b_2$ 에서 $\frac{\partial z_2^{(i)}}{\partial a_{1, j}^{(i)}} = W_{2, j, 1}$ 입니다.<br>
  출력층의 오차 dL_dz2 $(4, 1)$ 를 은닉층의 2개 노드로 다시 분배하여 $a_1$ 의 차원 $(4, 2)$ 를 맞추기 위해, $(2, 1)$ 행렬인 W2를 전치시켜 $(1, 2)$ 인 W2.T로 만든 뒤 행렬 곱을 수행합니다.
  $$\frac{\partial L}{\partial a_1} = \frac{\partial L}{\partial z_2} W_2^T \quad \Longrightarrow \quad \text{Shape: } (4, 1) \times (1, 2) = (4, 2)$$
  2. 은닉층 선형 출력 $z_1$ 미분 (dL_dz1): <br>
  활성화 함수 $a_1 = \sigma(z_1)$ 는 행렬의 각 원소에 독립적으로(Element-wise) 적용된 함수입니다.
  따라서 Sigmoid의 미분 공식 $\frac{\partial a_1}{\partial z_1} = a_1 \odot (1 - a_1)$ 을 구한 뒤, 앞에서 구한 $\frac{\partial L}{\partial a_1}$ 과 행렬 곱(@)이 아닌 원소별 곱셈(*, Hadamard product, $\odot$)을 수행합니다.
  $$\frac{\partial L}{\partial z_1} = \frac{\partial L}{\partial a_1} \odot a_1 \odot (1 - a_1) \quad \Longrightarrow \quad \text{Shape: } (4, 2) \odot (4, 2) \odot (4, 2) = (4, 2)$$
  ---
  #### Step 4: 은닉층 가중치 $W_1$ 과 편향 $b_1$ 미분 (dL_dW1, dL_db1)
* 대상 코드:
  ```python
  dL_dW1 = X.T @ dL_dz1
  dL_db1 = np.sum(dL_dz1, axis=0, keepdims=True)
  ```
* 연쇄 법칙(Chain Rule) 명시: <br>
순전파 식 $z_1 = X W_1 + b_1$ (원소 형태: $z_{1, j}^{(i)} = \sum_{m=1}^{2} X_{m}^{(i)} W_{1, m, j} + b_{1, j}$)에 대하여 연쇄 법칙을 적용합니다(전체 경로로 풀어서 쓰면 $\frac{\partial L}{\partial W_1} = \frac{\partial L}{\partial \hat{y}} \frac{\partial \hat{y}}{\partial z_2} \frac{\partial z_2}{\partial a_1} \frac{\partial a_1}{\partial z_1} \frac{\partial z_1}{\partial W_1}$ 입니다).
$$\frac{\partial L}{\partial W_{1, m, j}} = \sum_{i=1}^{N} \frac{\partial L}{\partial z_{1, j}^{(i)}} \cdot \frac{\partial z_{1, j}^{(i)}}{\partial W_{1, m, j}}, \qquad \frac{\partial L}{\partial b_{1, j}} = \sum_{i=1}^{N} \frac{\partial L}{\partial z_{1, j}^{(i)}} \cdot \frac{\partial z_{1, j}^{(i)}}{\partial b_{1, j}}$$
* 단계별 유도 및 전치 행렬 사용: <br>
  1. 가중치 $W_1$ 미분 (dL_dW1):
  $\frac{\partial z_{1, j}^{(i)}}{\partial W_{1, m, j}} = X_{m}^{(i)}$ 이므로, $\frac{\partial L}{\partial W_{1, m, j}} = \sum_{i=1}^{N} X_{m}^{(i)} \frac{\partial L}{\partial z_{1, j}^{(i)}}$ 가 됩니다. <br>
  Step 2의 dL_dW2와 완전히 동일한 원리로, 4개 샘플($i=1\dots 4$)에 대한 기울기를 합산하고 $W_1$ 의 차원 $(2, 2)$ 를 맞추기 위해 입력 행렬 X $(4, 2)$ 를 전치시켜 X.T $(2, 4)$ 로 만든 후 dL_dz1 $(4, 2)$ 과 행렬 곱을 수행합니다.
  2. 편향 $b_1$ 미분 (dL_db1): <br>
  $b_1$ $(1, 2)$ 역시 4개의 샘플에 브로드캐스팅되어 더해졌으므로($\frac{\partial z_{1, j}^{(i)}}{\partial b_{1, j}} = 1$), 샘플 차원(axis=0)을 따라 dL_dz1 $(4, 2)$ 의 값을 모두 합산하여 $(1, 2)$ 크기의 기울기 벡터를 얻습니다.

In [2]:
# 4. 학습 루프
for epoch in range(20000):
    # =========================================================================
    # Forward Propagation (순전파)
    # =========================================================================

    # [은닉층 선형 결합] z1 = X @ W1 + b1
    # Shape: (4, 2) @ (2, 2) + (1, 2)[브로드캐스팅] = (4, 2)
    z1 = X @ W1 + b1

    # [은닉층 활성화] a1 = sigmoid(z1) (원소별 연산)
    # Shape: (4, 2)
    a1 = sigmoid(z1)

    # [출력층 선형 결합] z2 = a1 @ W2 + b2
    # Shape: (4, 1) = (4, 2) @ (2, 1) + (1, 1)[브로드캐스팅]
    z2 = a1 @ W2 + b2

    # [출력층 활성화 (예측값)] y_hat = sigmoid(z2) (원소별 연산)
    # Shape: (4, 1)
    y_hat = sigmoid(z2)

    # [손실 함수 (Binary Cross Entropy)]
    # 수식: L = - (1/N) * sum( y * ln(y_hat) + (1 - y) * ln(1 - y_hat) )
    # Shape: 스칼라 (단일 값)
    loss = -np.mean(y * np.log(y_hat + 1e-8) + (1 - y) * np.log(1 - y_hat + 1e-8))

    # -------------------------------------------------------------------------
    # [Step 1] 출력층 선형 출력 z2에 대한 손실 미분 (dL_dz2)
    # -------------------------------------------------------------------------
    # * Chain Rule: dL/dz2 = (dL/dy_hat) * (dy_hat/dz2)
    #   1) dL/dy_hat = (1/N) * (y_hat - y) / (y_hat * (1 - y_hat))
    #   2) dy_hat/dz2 = y_hat * (1 - y_hat)  (Sigmoid 미분)
    #   3) 두 식을 곱하면 분모와 분자의 y_hat * (1 - y_hat) 항이 약분되어 소거됨
    # * 최종 수식: dL/dz2 = (y_hat - y) / N
    # * Shape: (4, 1) -> z2의 차원 (4, 1)과 동일
    dL_dz2 = (y_hat - y) / N

    # -------------------------------------------------------------------------
    # [Step 2] 출력층 가중치 W2와 편향 b2 미분 (dL_dW2, dL_db2)
    # -------------------------------------------------------------------------
    # [2-1. 가중치 W2 미분: dL_dW2]
    # * 순전파 수식: z2 = a1 @ W2 + b2  (원소: z2_{i,1} = sum_j(a1_{i,j} * W2_{j,1}) + b2_1)
    # * Chain Rule: dL/dW2_{j,1} = sum_{i=1}^N (dL/dz2_{i,1} * dz2_{i,1}/dW2_{j,1})
    #                            = sum_{i=1}^N (dL/dz2_{i,1} * a1_{i,j})
    dL_dW2 = a1.T @ dL_dz2

    # [2-2. 편향 b2 미분: dL_db2]
    # * Chain Rule: dL/db2 = sum_{i=1}^N (dL/dz2^{(i)} * dz2^{(i)}/db2) = sum_{i=1}^N (dL/dz2^{(i)} * 1)
    # * 순전파 때 b2(1, 1)가 4개 샘플에 브로드캐스팅되어 더해졌으므로, 역전파 시에는 샘플 차원(axis=0)으로
    #   오차를 모두 합산(np.sum)하며, keepdims=True로 원래 b2의 차원 (1, 1)을 유지함.
    dL_db2 = np.sum(dL_dz2, axis=0, keepdims=True)

    # -------------------------------------------------------------------------
    # [Step 3] 은닉층 활성화 출력 a1 및 선형 출력 z1 미분 (dL_da1, dL_dz1)
    # -------------------------------------------------------------------------
    # [3-1. 은닉층 활성화 값 a1 미분: dL_da1]
    # * 순전파 수식: z2 = a1 @ W2 + b2
    # * Chain Rule: dL/da1_{i,j} = sum_k (dL/dz2_{i,k} * dz2_{i,k}/da1_{i,j})
    #                            = dL/dz2_{i,1} * W2_{j,1}
    dL_da1 = dL_dz2 @ W2.T

    # [3-2. 은닉층 선형 출력 z1 미분: dL_dz1]
    # * 순전파 수식: a1 = sigmoid(z1) (각 원소별로 독립 적용된 함수)
    # * Chain Rule: dL/dz1 = (dL/da1) * (da1/dz1)
    #   여기서 Sigmoid 미분에 의해 da1/dz1 = a1 * (1 - a1)
    # * 원소별 연산의 역전파이므로 행렬 곱(@)이 아닌 원소별 곱셈(*, Hadamard product)을 수행함.
    # * Shape: (4, 2) * (4, 2) * (4, 2) = (4, 2) -> z1의 차원 (4, 2)와 동일
    dL_dz1 = dL_da1 * a1 * (1 - a1)

    # -------------------------------------------------------------------------
    # [Step 4] 은닉층 가중치 W1과 편향 b1 미분 (dL_dW1, dL_db1)
    # -------------------------------------------------------------------------
    # [4-1. 가중치 W1 미분: dL_dW1]
    # * 순전파 수식: z1 = X @ W1 + b1  (원소: z1_{i,j} = sum_m(X_{i,m} * W1_{m,j}) + b1_j)
    # * 전체 Chain Rule 경로: dL/dW1 = (dL/dy_hat) * (dy_hat/dz2) * (dz2/da1) * (da1/dz1) * (dz1/dW1)
    #                               = dL_dz1과 dz1/dW1(= X)의 결합
    # * 원소별 수식: dL/dW1_{m,j} = sum_{i=1}^N (dL/dz1_{i,j} * dz1_{i,j}/dW1_{m,j})
    #                            = sum_{i=1}^N (X_{i,m} * dL/dz1_{i,j})
    dL_dW1 = X.T @ dL_dz1

    # [4-2. 편향 b1 미분: dL_db1]
    # * Chain Rule: dL/db1 = sum_{i=1}^N (dL/dz1^{(i)} * dz1^{(i)}/db1) = sum_{i=1}^N (dL/dz1^{(i)} * 1)
    # * 브로드캐스팅된 편향의 역전파이므로 샘플 차원(axis=0)으로 dL_dz1(4, 2)을 합산하여 (1, 2) 차원으로 만듦.
    dL_db1 = np.sum(dL_dz1, axis=0, keepdims=True)

    # =========================================================================
    # Gradient Descent (경사 하강법을 통한 파라미터 업데이트)
    # =========================================================================
    # 각 파라미터와 해당 기울기(dL_dW, dL_db)의 Shape이 정확히 일치하므로 뺄셈 연산이 성립함
    W2 -= lr * dL_dW2  # (2, 1) -= 스칼라 * (2, 1)
    b2 -= lr * dL_db2  # (1, 1) -= 스칼라 * (1, 1)

    W1 -= lr * dL_dW1  # (2, 2) -= 스칼라 * (2, 2)
    b1 -= lr * dL_db1  # (1, 2) -= 스칼라 * (1, 2)

    if epoch % 2000 == 0:
        print(f"Epoch {epoch}, Loss: {loss:.4f}")

Epoch 0, Loss: 0.2742
Epoch 2000, Loss: 0.0048
Epoch 4000, Loss: 0.0024
Epoch 6000, Loss: 0.0016
Epoch 8000, Loss: 0.0012
Epoch 10000, Loss: 0.0009
Epoch 12000, Loss: 0.0008
Epoch 14000, Loss: 0.0007
Epoch 16000, Loss: 0.0006
Epoch 18000, Loss: 0.0005


### 4. 최종 학습 결과 확인

In [3]:
print("\nX:")
print(X)
print("\n예측 결과:")
print(np.round(y_hat, 3))
print("\n분류 결과:")
print((y_hat >= 0.5).astype(int))
print("\nY:")
print(y)


X:
[[0. 0.]
 [0. 1.]
 [1. 0.]
 [1. 1.]]

예측 결과:
[[0.001]
 [1.   ]
 [1.   ]
 [0.   ]]

분류 결과:
[[0]
 [1]
 [1]
 [0]]

Y:
[[0.]
 [1.]
 [1.]
 [0.]]
